# JobSpy scraper

Parameters below are read from environment variables so this notebook can be run
unattended by the `scrape-jobs.yml` GitHub Actions workflow. When run manually,
just edit the defaults.

In [ ]:
import os

search_term = os.environ.get("SEARCH_TERM", "data analyst")
google_search_term = os.environ.get("GOOGLE_SEARCH_TERM", f"{search_term} jobs")
location = os.environ.get("LOCATION", "Singapore")
country_indeed = os.environ.get("COUNTRY_INDEED", "Singapore")
site_names = os.environ.get("SITE_NAMES", "indeed,linkedin,google").split(",")
results_wanted = int(os.environ.get("RESULTS_WANTED", "100"))
output_csv = os.environ.get("OUTPUT_CSV", "jobs.csv")

# Optional filters. Per JobSpy's own limitations (see README), Indeed and LinkedIn
# each only honor one of hours_old / job_type+is_remote / easy_apply at a time —
# whichever is set takes effect per-site, the others are simply ignored by that site.
hours_old_raw = os.environ.get("HOURS_OLD", "168")
hours_old = int(hours_old_raw) if hours_old_raw else None

job_type = os.environ.get("JOB_TYPE", "") or None

is_remote_raw = os.environ.get("IS_REMOTE", "").lower()
is_remote = is_remote_raw == "true"

easy_apply_raw = os.environ.get("EASY_APPLY", "").lower()
easy_apply = True if easy_apply_raw == "true" else None

distance_raw = os.environ.get("DISTANCE", "")
distance = int(distance_raw) if distance_raw else None

print(f"search_term={search_term!r} location={location!r} sites={site_names}")
print(
    f"job_type={job_type!r} is_remote={is_remote!r} easy_apply={easy_apply!r} "
    f"distance={distance!r} hours_old={hours_old!r}"
)

In [ ]:
import csv
from jobspy import scrape_jobs


jobs = scrape_jobs(
    site_name=site_names,
    search_term=search_term,
    google_search_term=google_search_term,
    location=location,
    results_wanted=results_wanted,
    hours_old=hours_old,
    country_indeed=country_indeed,
    job_type=job_type,
    is_remote=is_remote,
    easy_apply=easy_apply,
    distance=distance,

    # linkedin_fetch_description=True # gets more info such as description, direct job url (slower)
    # proxies=["208.195.175.46:65095", "208.195.175.45:65095", "localhost"],
)
print(f"Found {len(jobs)} jobs")
print(jobs.head())
jobs.to_csv(output_csv, quoting=csv.QUOTE_NONNUMERIC, escapechar="\\", index=False)